In [13]:
import akshare as ak
import pandas as pd
import os,re
import time
import random

def download_all_profits():
    # 1. 设置保存数据的文件夹路径 (建议新建一个单独的文件夹存放这5000多个文件)
    save_dir = r"C:\data\利润"

    # 如果文件夹不存在，则自动创建
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)

    # 2. 获取A股所有股票的基本信息（包含股票代码和名称）
    print("正在获取A股所有股票代码列表...")
    stock_info_df = ak.stock_info_a_code_name()


    # #退市股票
    # # 获取沪市退市名单
    # df_sh_delist = ak.stock_info_sh_delist()
    # df_sh_delist = df_sh_delist[['公司代码', '公司简称']]
    # df_sh_delist.columns = ['code', 'name'] # 统一列名
    # # 获取深市退市名单
    # df_sz_delist = ak.stock_info_sz_delist()
    # df_sz_delist = df_sz_delist[['证券代码', '证券简称']]
    # df_sz_delist.columns = ['code', 'name'] # 统一列名
    # #合并沪深退市名单
    # stock_info_df = pd.concat([df_sh_delist, df_sz_delist], ignore_index=True)


    total_stocks = len(stock_info_df)
    print(f"共获取到 {total_stocks} 只股票，准备开始批量下载...\n")
    # 3. 遍历所有股票代码
    for index, row in stock_info_df.iterrows():
        # 获取原始代码，并确保是6位字符串（如 000001）
        base_code = str(row['code']).zfill(6)
        stock_name = row['name']

        # 转换成新浪接口需要的格式 (sh: 沪市, sz: 深市, bj: 北交所)
        if base_code.startswith(('60', '68', '69')):
            sina_symbol = f"sh{base_code}"
        elif base_code.startswith(('00', '30')):
            sina_symbol = f"sz{base_code}"
        elif base_code.startswith(('4', '8', '9')):
            sina_symbol = f"bj{base_code}"
        else:
            continue # 其他未知代码跳过

        safe_stock_name = re.sub(r'[\\/:*?"<>|]', '', stock_name)
        # 定义该股票 CSV 的完整保存路径 (文件名带上股票名称方便查看)
        file_path = os.path.join(save_dir, f"{sina_symbol}_{safe_stock_name}_利润表.csv")

        # 【重点】断点续传逻辑：如果该股票的CSV已经存在，则直接跳过，不用重新下载
        if os.path.exists(file_path):
            print(f"[{index + 1}/{total_stocks}] {sina_symbol} {stock_name} 已存在，跳过...")
            continue

        # 4. 尝试下载数据
        try:
            print(f"[{index + 1}/{total_stocks}] 正在下载 {sina_symbol} {stock_name} 的利润表...")

            # 调用接口
            df = ak.stock_financial_report_sina(stock=sina_symbol, symbol="利润表")

            # 如果成功获取到数据且不为空，则保存
            if df is not None and not df.empty:
                # 依然使用gbk编码，兼容Windows下的Excel直接打开
                df.to_csv(file_path, encoding="gbk", index=False)
                print(f"    => 下载成功！")
            else:
                print(f"    => 数据为空 (可能是新股或无财报数据)！")

        except Exception as e:
            # 捕获异常，防止某一只股票报错导致整个程序崩溃
            print(f"    => 下载失败，错误信息: {e}")

        # 【重点】防封IP延时：每次下载完随机休息 1 到 2 秒
        # 批量下载切忌太快，否则新浪会封锁你的IP导致全部报 timeout
        time.sleep(random.uniform(2, 3))

if __name__ == "__main__":
    download_all_profits()
    print("\n全部任务执行完毕！")

正在获取A股所有股票代码列表...
共获取到 5493 只股票，准备开始批量下载...

[1/5493] sz000001 平安银行 已存在，跳过...
[2/5493] sz000002 万  科Ａ 已存在，跳过...
[3/5493] sz000004 *ST国华 已存在，跳过...
[4/5493] sz000006 深振业Ａ 已存在，跳过...
[5/5493] sz000007 全新好 已存在，跳过...
[6/5493] sz000008 神州高铁 已存在，跳过...
[7/5493] sz000009 中国宝安 已存在，跳过...
[8/5493] sz000010 美丽生态 已存在，跳过...
[9/5493] sz000011 深物业A 已存在，跳过...
[10/5493] sz000012 南  玻Ａ 已存在，跳过...
[11/5493] sz000014 沙河股份 已存在，跳过...
[12/5493] sz000016 深康佳Ａ 已存在，跳过...
[13/5493] sz000017 深中华A 已存在，跳过...
[14/5493] sz000019 深粮控股 已存在，跳过...
[15/5493] sz000020 深华发Ａ 已存在，跳过...
[16/5493] sz000021 深科技 已存在，跳过...
[17/5493] sz000025 特  力Ａ 已存在，跳过...
[18/5493] sz000026 飞亚达 已存在，跳过...
[19/5493] sz000027 深圳能源 已存在，跳过...
[20/5493] sz000028 国药一致 已存在，跳过...
[21/5493] sz000029 深深房Ａ 已存在，跳过...
[22/5493] sz000030 富奥股份 已存在，跳过...
[23/5493] sz000031 大悦城 已存在，跳过...
[24/5493] sz000032 深桑达Ａ 已存在，跳过...
[25/5493] sz000034 神州数码 已存在，跳过...
[26/5493] sz000035 中国天楹 已存在，跳过...
[27/5493] sz000036 华联控股 已存在，跳过...
[28/5493] sz000037 深南电A 已存在，跳过...
[29/5493] sz